# Factual Consistency Checking (FactScore Method)

In [ ]:
# !pip install --upgrade factscore
# !python -m spacy download en_core_web_sm

# Prepare dataset

In [ ]:
import pandas as pd
df = pd.read_csv('../datasets.csv')
print(df.shape)
display(df)

In [ ]:
# drop error row
df.loc[161]

In [ ]:
df = df.drop(161).reset_index(drop=True)
print(df.shape)
display(df)

# Build pipeline to compare AI_essay and humanized_essay using FactScore

In [ ]:
df['factscore_forward'] = 0
df['factscore_reversed'] = 0
df.head()

In [ ]:
processed_ai_text = set()

In [ ]:
import os
import json
import tempfile
from factscore.factscorer import FactScorer

topic_map = {1: "Effects of social media on youth", 2: "How to fix a broken fan", 3: "Summary of harry potter and the philosopher stone", 4: "The ethics of artificial intelligence", 5: "My favourite childhood cartoon"}


def split_into_paragraphs(text):
    """Split text into list of paragraphs for better retrieval."""
    return [para.strip() for para in text.split('\n\n') if para.strip()]

def create_jsonl_kb(essay_text, topic, file_path):
    """Create .jsonl file for custom knowledge base."""
    data = {"title": topic, "text": split_into_paragraphs(essay_text)}
    with open(file_path, 'w') as f:
        try:
            json.dump(data, f)
            f.write('\n')  # .jsonl format (one line)
        except TypeError as e:
            print(f"Error writing JSON: {e}")
            print(f"Data that caused error: {data}")

def evaluate_essays(AI_essay, humanized_essay, topic_id, factscorer, gamma=10):
    topic = topic_map.get(topic_id, f"Topic {topic_id}")
    with tempfile.TemporaryDirectory() as tmp_dir:
        kb_a_path = os.path.join(tmp_dir, "kb_a.jsonl")
        kb_b_path = os.path.join(tmp_dir, "kb_b.jsonl")
        
        create_jsonl_kb(humanized_essay, topic, kb_b_path)
        
        # Register custom KBs (this builds retrieval indices if needed)
        factscorer.register_knowledge_source(name="essayA", data_path=kb_a_path)
        factscorer.register_knowledge_source(name="essayB", data_path=kb_b_path)
        
        # Score B against A (preservation in paraphrase + detect new facts)
        out_b_vs_a = factscorer.get_score(topics=[topic], generations=[humanized_essay], gamma=gamma, knowledge_source="essayA")
        score_b_vs_a = out_b_vs_a["score"][0]
        decisions_b = out_b_vs_a["decisions"][0]
        new_facts_b = [d["claim"] for d in decisions_b if d["decision"] == "no"]  # Unintended/new facts
        
        # Score A against B (for symmetry, e.g., omissions in B)
        out_a_vs_b = factscorer.get_score(topics=[topic], generations=[AI_essay], gamma=gamma, knowledge_source="essayB")
        score_a_vs_b = out_a_vs_b["score"][0]
        decisions_a = out_a_vs_b["decisions"][0]
        unsupported_in_a = [d["claim"] for d in decisions_a if d["decision"] == "no"]  # Facts in A not in B
        
        # Print results
        print(f"FActScore of B against A: {score_b_vs_a:.2f}% (preservation ratio)")
        print(f"Number of atomic facts in B: {len(decisions_b)}")
        if new_facts_b:
            print("New/unintended facts introduced in B:")
            for fact in new_facts_b:
                print(f"- {fact}")
        else:
            print("No new/unintended facts detected in B.")
        
        print(f"\nFActScore of A against B: {score_a_vs_b:.2f}% (coverage by paraphrase)")
        print(f"Number of atomic facts in A: {len(decisions_a)}")
        if unsupported_in_a:
            print("Facts in A not supported by B (e.g., omissions/alterations):")
            for fact in unsupported_in_a:
                print(f"- {fact}")
        else:
            print("All facts in A are supported by B.")

In [ ]:
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")

In [ ]:
from factscore.factscorer import FactScorer
fs = FactScorer(openai_key=API_KEY)  # Uses retrieval+ChatGPT by default

In [ ]:
# try run for first row
AI_essay = df.loc[0, 'AI_essay']
humanized_essay = df.loc[0, 'humanized_essay']
topic_id = df.loc[0, 'topic']
evaluate_essays(AI_essay, humanized_essay, topic_id, fs)